# Détection (fenêtre glissante) + comparaison ResNet18 / ResNet50 - Colab

**Avant de commencer :** menu *Exécution > Modifier le type d'exécution > GPU (T4)*.

À la racine de *Mon Drive* il faut : `projet_avions.zip`, `projet_avions_detection_v2.zip` et `projet_avions_code.zip`.

Chaque cellule **sauvegarde ses résultats sur Drive** et **saute ce qui est déjà fait** : si Colab se déconnecte, relance la cellule 1 puis reprends là où tu en étais.

In [ ]:
# 1. GPU, Drive, décompression des données + du code, récupération des résultats déjà sauvegardés
!nvidia-smi --query-gpu=name --format=csv
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive'
R = D + '/resultats_avions'
!unzip -q -o {D}/projet_avions.zip -d /content/
!unzip -q -o {D}/projet_avions_detection_v2.zip -d /content/
!unzip -q -o {D}/projet_avions_code.zip -d /content/
%cd /content/projet_avions
!mkdir -p outputs/classification outputs/detection {R}/classification {R}/detection
!cp -r {R}/resnet50 outputs/classification/ 2>/dev/null; cp -r {R}/classification/* outputs/classification/ 2>/dev/null; cp -r {R}/detection/* outputs/detection/ 2>/dev/null
!ls data outputs/classification outputs/detection

In [ ]:
# 2. Détecteur ResNet18 : entraînement avion/fond (~10 min) puis détection sur les images (~20 min)
import os, shutil
%cd /content/projet_avions
R = '/content/drive/MyDrive/resultats_avions'
def sauver(d):  # copie outputs/<d> vers Drive
    shutil.copytree('outputs/' + d, R + '/' + d, dirs_exist_ok=True); print('sauvegardé sur Drive :', d)
N = 'detecteur_resnet18'
if not os.path.exists(f'outputs/classification/{N}/metrics.json'):
    !python 03_classification.py --data data/fenetres --arch resnet18 --size 96 --batch 128 --lr 1e-3 --epochs 12 --patience 4 --name $N
sauver('classification/' + N)
# détection : la partie lente (scores des fenêtres) n'est faite qu'une fois, ensuite elle est relue du cache
!python 06_detection.py --name $N
sauver('detection/' + N)
print(open(f'outputs/detection/{N}/rapport_detection.txt').read())

In [ ]:
# 3. Détecteur ResNet50 : entraînement avion/fond (~25 min) puis détection sur les images (~50 min)
import os, shutil
%cd /content/projet_avions
R = '/content/drive/MyDrive/resultats_avions'
def sauver(d):  # copie outputs/<d> vers Drive
    shutil.copytree('outputs/' + d, R + '/' + d, dirs_exist_ok=True); print('sauvegardé sur Drive :', d)
N = 'detecteur_resnet50'
if not os.path.exists(f'outputs/classification/{N}/metrics.json'):
    !python 03_classification.py --data data/fenetres --arch resnet50 --size 96 --batch 128 --lr 1e-3 --epochs 12 --patience 4 --name $N
sauver('classification/' + N)
# détection : la partie lente (scores des fenêtres) n'est faite qu'une fois, ensuite elle est relue du cache
!python 06_detection.py --name $N
sauver('detection/' + N)
print(open(f'outputs/detection/{N}/rapport_detection.txt').read())

In [ ]:
# 4. Classifieur 20 classes en ResNet18 (le ResNet50 est déjà fait) ~10 min
import os, shutil
%cd /content/projet_avions
R = '/content/drive/MyDrive/resultats_avions'
if not os.path.exists('outputs/classification/resnet18/metrics.json'):
    !python 03_classification.py --arch resnet18 --epochs 25
shutil.copytree('outputs/classification/resnet18', R + '/classification/resnet18', dirs_exist_ok=True)
print(open('outputs/classification/resnet18/rapport_test.txt').read()[:400])

In [ ]:
# 5. Comparaison ResNet18 / ResNet50
import os, shutil
from IPython.display import Image, display
%cd /content/projet_avions
R = '/content/drive/MyDrive/resultats_avions'
!python 07_comparaison.py
display(Image('outputs/comparaison/comparaison.png', width=1000))
shutil.copytree('outputs/comparaison', R + '/comparaison', dirs_exist_ok=True)

In [ ]:
# 6. Exemples de détection côte à côte (vert = correct, rouge = fausse alarme, jaune = avion manqué)
import os
import glob
for f in sorted(glob.glob('outputs/detection/detecteur_resnet50/exemples/*.jpg'))[:4]:
    nom = os.path.basename(f)
    print('Image', nom, ': ResNet18 (1re) puis ResNet50 (2e)')
    display(Image(f'outputs/detection/detecteur_resnet18/exemples/{nom}', width=600))
    display(Image(f, width=600))

In [ ]:
# 7. Corpus de test annoté par le modèle (pour le rapport) ~25 min
import os, shutil
%cd /content/projet_avions
R = '/content/drive/MyDrive/resultats_avions'
!python 09_corpus_annote.py
shutil.copy('outputs/corpus_annote.zip', R + '/corpus_annote.zip')
print('Copié sur Drive : resultats_avions/corpus_annote.zip')

In [ ]:
# 8. Application Streamlit : clique sur le lien https://....trycloudflare.com qui s'affiche
# (Colab n'ouvre pas directement les pages web : cloudflared crée un lien public temporaire)
import os, re, subprocess
%cd /content/projet_avions
!pip install -q streamlit
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
app = subprocess.Popen(['streamlit', 'run', '08_application.py', '--server.port', '8501', '--server.headless', 'true'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                          stderr=subprocess.PIPE, text=True)
for ligne in tunnel.stderr:
    lien = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', ligne)
    if lien:
        print('Application prête : ouvre ce lien ->', lien.group(0))
        print("(si la page dit qu'elle est introuvable, attends 20 secondes et recharge)")
        break
import threading   # continuer à lire les messages du tunnel en arrière-plan (sinon il peut se bloquer)
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()